# 真实本地模型：CAD 上下文感知解码

在知识冲突场景下运行 CAD：对每个 token 同时维护“有上下文/无上下文”两条 KV cache，按 (1+α)·logit_ctx − α·logit_no_ctx 组合，抑制与上下文冲突的参数化先验。

## 真实集成前提

安装：`python3 -m pip install -e ".[hf]"`。CAD 需要显式上下文；每个 token 两次前向，延迟与显存近似翻倍。CLI：`hallucination-mitigation run --method cad --context ... --prompt ...`。

代码单元没有预执行。填写本地模型路径或 API 凭据后，从上到下运行；不要将 API key 保存到 notebook。缓解会改变模型输出，务必同时阅读基线与缓解后的回答。

## 参数说明

- `alpha: float`：对比强度，0 退化为常规解码，越大越依赖上下文。
- `context_template: str = 'plain'`：上下文拼接模板（plain/chat）。
- `context`：与模型先验冲突的上下文事实，运行时交互输入。
- `MAX_NEW_TOKENS: int = 64`：生成长度。

## 步骤 1：配置真实模型

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

LOCAL_MODEL = "/absolute/path/to/your/causal-lm"   # e.g. a local Qwen2.5-1.5B-Instruct directory
DEVICE = "cpu"     # "cuda" when a GPU is available
DTYPE = "auto"
MAX_NEW_TOKENS = 64
if not Path(LOCAL_MODEL).is_dir():
    raise ValueError("Set LOCAL_MODEL to an existing local model directory (download it first).")

## 步骤 2：构建 Mitigator

In [ ]:
from hallucination.mitigation import Mitigator
mitigator = Mitigator(
    method="cad",
    model=LOCAL_MODEL,
    device=DEVICE,
    dtype=DTYPE,
    local_files_only=True,
    alpha=1.0,
    context_template="plain",
    baseline=True,
)

## 步骤 3：交互运行并读取

In [ ]:
context = input("上下文事实（与模型先验冲突时效果最明显，如“阿根廷在 1978、1986 和 2022 年赢得世界杯”）：").strip()
prompt = input("问题：").strip()
report = mitigator.mitigate(prompt, context=context, max_new_tokens=32)
print("baseline :", report.baseline)
print("mitigated:", report.response)
print("warnings:", report.warnings)

## 结果解读与限制

CAD 只在上下文与先验冲突时有收益；无冲突场景可能过度修正。α 需要按任务验证，报告中的 KL 轨迹可辅助诊断。